# 1. Montar Google Drive e Importaciones

In [1]:
# ==========================================
# 1. MONTAR GOOGLE DRIVE E INSTALAR DEPENDENCIAS
# ==========================================
from google.colab import drive
import os
import json
import orjson
import pandas as pd
import gc
from collections import defaultdict
import shutil


drive.mount('/content/drive')

Mounted at /content/drive


# 2. Configuración de Rutas

In [2]:
# ==========================================
# 2. CONFIGURACIÓN DE RUTAS SEGÚN TUS CARPETAS
# ==========================================
BASE_DRIVE = '/content/drive/MyDrive/TFG/data'

DIR_PEMR = os.path.join(BASE_DRIVE, 'PatientEncounterMedRequest')
DIR_VARIOS = os.path.join(BASE_DRIVE, 'Varios')
DIR_REST = os.path.join(BASE_DRIVE, 'Rest')

# Carpeta local temporal en Colab para guardar el resultado de salida sin llenar tu PC
OUTPUT_DIR = '/content/output_tfg'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'medication_request_dataset_full.jsonl')

# Rutas de los archivos distribuidos en tus 3 carpetas
PATIENT_FILE = os.path.join(DIR_PEMR, 'MimicPatient.ndjson')
MED_REQUEST_FILE = os.path.join(DIR_PEMR, 'MimicMedicationRequest.ndjson')
ENCOUNTER_FILES = [
    os.path.join(DIR_PEMR, 'MimicEncounter.ndjson'),
    os.path.join(DIR_REST, 'MimicEncounterED.ndjson')
]
CONDITION_FILES = [
    os.path.join(DIR_VARIOS, 'MimicCondition.ndjson'),
    os.path.join(DIR_REST, 'MimicConditionED.ndjson')
]
MED_FILE = os.path.join(DIR_VARIOS, 'MimicMedication.ndjson')
ADMIN_FILE = os.path.join(DIR_REST, 'MimicMedicationAdministration.ndjson')
PROCEDURE_FILES = [
    os.path.join(DIR_VARIOS, 'MimicProcedure.ndjson')
]

# El CSV de prescripciones con los identificadores de médicos
CSV_MEDICOS = os.path.join(BASE_DRIVE, 'prescriptions.csv.gz')

# Ruta del CSV de pacientes con los slots temporales (para filtrar el sesgo de anonimización)
PATIENTS_CSV_FILE = os.path.join(BASE_DRIVE, 'patients.csv')

# 3. Herramientas Base (Iterador NDJSON y Función Segura *dig*)

In [3]:
# ==========================================
# 3. HERRAMIENTAS BASE
# ==========================================

def iter_ndjson(path, stats):
    """
    Lee los archivos línea a línea de forma eficiente y los convierte de texto
    plano a diccionarios de Python usando orjson, incluyendo control de errores.
    """
    if not os.path.exists(path):
        print(f"⚠️ Aviso: No se encontró {path}")
        return
    with open(path, 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                yield orjson.loads(line)
            except (orjson.JSONDecodeError, ValueError):
                stats['json_errors'] += 1

def dig(d, *path, default=None):
    """
    Función de seguridad para trabajar con estándares FHIR.
    Evita KeyError cuando los datos anidados profundos no existen,
    devolviendo un valor por defecto de manera limpia.
    """
    for k in path:
        try:
            d = d[k]
        except (KeyError, IndexError, TypeError):
            return default
    return d

# 4. Funciones de Carga en RAM (Dimensiones y Mapeo Temporal)

In [4]:
# ==========================================
# 4. FUNCIONES DE CARGA OPTIMIZADAS (FILTRADO EN ORIGEN)
# ==========================================
import gc

def cargar_pacientes(filepath, mapa_slots, stats):
    """
    Carga únicamente los pacientes que pertenecen estrictamente al slot 2008-2010.
    """
    print("⏳ Cargando y seleccionando pacientes (Slot 2008-2010)...")
    diccionario = {}
    pacientes_validos = set()

    for paciente in iter_ndjson(filepath, stats):
        fhir_uuid = paciente.get('id')

        # Extraer subject_id numérico
        subject_id_num = None
        for ident in paciente.get('identifier', []):
            if 'mimic/identifier/patient' in ident.get('system', ''):
                subject_id_num = str(ident.get('value'))
                break

        slot = mapa_slots.get(subject_id_num, 'Desconocido')

        # Filtro estricto para el slot 2008-2010
        if slot == '2008 - 2010':
            pacientes_validos.add(fhir_uuid)
            diccionario[fhir_uuid] = {
                'subject_id': subject_id_num,
                'anchor_year_group': slot,
                'patient_gender': paciente.get('gender', 'unknown'),
                'patient_birthdate': paciente.get('birthDate', 'unknown')
            }

    print(f"✅ Pacientes cargados en RAM: {len(diccionario):,}")
    return diccionario, pacientes_validos


def cargar_encounters(filepaths, pacientes_validos_ids, stats):
    """Carga solo los encuentros (ingresos) de nuestros pacientes válidos."""
    print("⏳ Cargando ingresos hospitalarios...")
    diccionario = {}
    encuentros_validos = set()

    for filepath in filepaths:
        for enc in iter_ndjson(filepath, stats):
            pat_ref = dig(enc, 'subject', 'reference', default='')
            pat_id = pat_ref.rsplit('/', 1)[-1] if '/' in pat_ref else ''

            if pat_id in pacientes_validos_ids:
                enc_id = enc.get('id')
                encuentros_validos.add(enc_id)
                diccionario[enc_id] = {
                    'encounter_class': dig(enc, 'class', 'code', default='unknown'),
                    'encounter_start': dig(enc, 'period', 'start', default=''),
                    'encounter_end': dig(enc, 'period', 'end', default=''),
                    'encounter_service_type': dig(enc, 'serviceType', 'coding', 0, 'code', default='unknown')
                }
    print(f"✅ Encuentros cargados: {len(diccionario):,}")
    return diccionario, encuentros_validos


def cargar_diagnosticos(filepaths, encuentros_validos_ids, stats):
    """Carga solo los diagnósticos de los encuentros de nuestra cohorte."""
    print("⏳ Cargando diagnósticos...")
    por_enc = defaultdict(dict)
    for filepath in filepaths:
        for cond in iter_ndjson(filepath, stats):
            enc_ref = dig(cond, 'encounter', 'reference', default='')
            enc_id = enc_ref.rsplit('/', 1)[-1] if '/' in enc_ref else ''

            if enc_id in encuentros_validos_ids:
                cod = dig(cond, 'code', 'coding', 0, 'code', default='UNKNOWN')
                txt = dig(cond, 'code', 'coding', 0, 'display', default='No description')
                por_enc[enc_id][cod] = txt
    return por_enc


def cargar_procedimientos(filepaths, encuentros_validos_ids, stats):
    """Carga solo los procedimientos de los encuentros de nuestra cohorte."""
    print("⏳ Cargando procedimientos...")
    por_enc = defaultdict(dict)
    for filepath in filepaths:
        for proc in iter_ndjson(filepath, stats):
            enc_ref = dig(proc, 'encounter', 'reference', default='')
            if not enc_ref:
                enc_ref = proc.get('encounter', '')
                if isinstance(enc_ref, dict):
                    enc_ref = enc_ref.get('reference', '')
            enc_id = enc_ref.rsplit('/', 1)[-1] if '/' in enc_ref else str(enc_ref)

            if enc_id in encuentros_validos_ids:
                cod = dig(proc, 'code', 'coding', 0, 'code', default='UNKNOWN')
                txt = dig(proc, 'code', 'coding', 0, 'display', default='No description')
                por_enc[enc_id][cod] = txt
    return por_enc


def cargar_medicamentos(filepath, stats):
    """Carga el catálogo de medicamentos."""
    print("⏳ Cargando catálogo de medicamentos...")
    diccionario = {}
    for med in iter_ndjson(filepath, stats):
        med_id = med.get('id')
        med_name = "unknown"
        for ident in med.get('identifier', []):
            if 'mimic-medication-name' in ident.get('system', ''):
                med_name = ident.get('value', 'unknown')
                break
        if med_name == "unknown":
            med_name = dig(med, 'code', 'text', default="unknown")
        diccionario[med_id] = med_name
    return diccionario


def cargar_administraciones(filepath, stats):
    """Carga los registros de administración."""
    print("⏳ Cargando registros de administración...")
    administraciones = set()
    for admin in iter_ndjson(filepath, stats):
        req_ref = dig(admin, 'request', 'reference', default='')
        req_id = req_ref.rsplit('/', 1)[-1] if '/' in req_ref else ''
        if req_id:
            administrations_add = administraciones.add(req_id)
    return administraciones

# 5. Procesamiento Maestro (ETL y Filtrado del Slot 2008-2010)

In [5]:
# ==============================================================================
# PIPELINE ETL COMPLETO Y UNIFICADO - FILTRADO ESTRICTO AÑO 2008
# ==============================================================================

print("="*80)
print(" 🚀 INICIANDO PROCESO ETL UNIFICADO (COHORTE 2008-2010 -> FILTRO 2008)")
print("="*80)

# 1. Cargar el mapa de médicos
print("\n⏳ [1/5] Cargando el mapa de médicos...")
if not os.path.exists(CSV_MEDICOS):
    raise FileNotFoundError(f"❌ No se encuentra el archivo CSV en {CSV_MEDICOS}.")

df_med = pd.read_csv(CSV_MEDICOS, usecols=['pharmacy_id', 'order_provider_id'], dtype=str)
mapa_medicos = pd.Series(df_med.order_provider_id.values, index=df_med.pharmacy_id.astype(str)).to_dict()
print(f"✅ Mapa de médicos cargado: {len(mapa_medicos):,} registros indexados.")

# 2. Cargar el mapa de slots de pacientes
print("\n⏳ [2/5] Cargando el mapa de slots de pacientes...")
if not os.path.exists(PATIENTS_CSV_FILE):
    raise FileNotFoundError(f"❌ No se encuentra patients.csv en {PATIENTS_CSV_FILE}.")

df_pat = pd.read_csv(PATIENTS_CSV_FILE, dtype={'subject_id': str})
mapa_slots = dict(zip(df_pat['subject_id'], df_pat['anchor_year_group']))
print(f"✅ Mapa de slots cargado: {len(mapa_slots):,} pacientes indexados.")

stats = defaultdict(int)

# 3. Cargar dimensiones filtradas
print("\n⏳ [3/5] Cargando dimensiones en RAM (Pacientes, Encuentros, Diagnósticos, etc.)...")
pacientes, pacientes_validos_ids = cargar_pacientes(PATIENT_FILE, mapa_slots, stats)
encounters, encuentros_validos_ids = cargar_encounters(ENCOUNTER_FILES, pacientes_validos_ids, stats)
diagnosticos = cargar_diagnosticos(CONDITION_FILES, encuentros_validos_ids, stats)
procedimientos = cargar_procedimientos(PROCEDURE_FILES, encuentros_validos_ids, stats)
medicamentos = cargar_medicamentos(MED_FILE, stats)
administraciones =_admin = cargar_administraciones(ADMIN_FILE, stats)

gc.collect()

import os
import pandas as pd
import orjson

CACHE_DELTAS_FILE = os.path.join(BASE_DRIVE, 'deltas_pacientes_cache.csv')

print("\n⏳ [4/5] Analizando fechas mínimas por paciente (Optimizado con Caché)...")

if os.path.exists(CACHE_DELTAS_FILE):
    print("⚡ ¡Caché encontrada! Cargando deltas instantáneamente desde Google Drive...")
    df_cache = pd.read_csv(CACHE_DELTAS_FILE, dtype={'patient_id': str})
    deltas_paciente = {
        row['patient_id']: pd.Timedelta(row['delta_ns'])
        for _, row in df_cache.iterrows()
    }
    print(f"✅ {len(deltas_paciente):,} deltas cargados en un par de segundos.")

else:
    print("⏳ Analizando archivo grande por primera vez (usando orjson de alta velocidad)...")
    primeras_fechas_paciente = {}

    with open(MED_REQUEST_FILE, 'rt', encoding='utf-8', errors='ignore') as f_temp:
        for i, linea in enumerate(f_temp):
            linea_limpia = linea.strip()
            if not linea_limpia: continue
            try:
                # Usamos orjson que es hiperrápido comparado con el json estándar
                receta = orjson.loads(linea_limpia)
            except:
                continue

            patient_id = dig(receta, 'subject', 'reference', default='').rsplit('/', 1)[-1]
            if patient_id in pacientes_validos_ids:
                authored = receta.get('authoredOn')
                if authored:
                    dt = pd.to_datetime(authored, errors='coerce')
                    if not pd.isnull(dt):
                        if patient_id not in primeras_fechas_paciente or dt < primeras_fechas_paciente[patient_id]:
                            primeras_fechas_paciente[patient_id] = dt

            if i > 0 and i % 500000 == 0:
                print(f"   -> Escaneadas {i:,} líneas...")

    deltas_paciente = {}
    for pat_id, primera_dt in primeras_fechas_paciente.items():
        try:
            objetivo_dt = primera_dt.replace(year=2008)
        except ValueError:
            objetivo_dt = primera_dt.replace(year=2008, day=28)
        deltas_paciente[pat_id] = primera_dt - objetivo_dt

    # 💾 Guardamos en caché para el futuro
    print("💾 Guardando deltas en caché en Google Drive...")
    datos_para_cache = [
        {'patient_id': pid, 'delta_ns': delta.value}
        for pid, delta in deltas_paciente.items()
    ]
    pd.DataFrame(datos_para_cache).to_csv(CACHE_DELTAS_FILE, index=False)
    print(f"✅ Caché guardada con éxito.")

print(f"✅ Deltas listos para {len(deltas_paciente):,} pacientes.")

# 5. Bucle principal de ETL (Escritura local rápida + Filtro estricto 2008)
print("\n🚀 [5/5] Iniciando bucle principal de transformación (Filtrando estrictamente el año 2008)...")

OUTPUT_LOCAL = '/content/medication_request_2008.jsonl'
contador_exito = 0
contador_total = 0

with open(MED_REQUEST_FILE, 'rt', encoding='utf-8', errors='ignore') as f_in, \
     open(OUTPUT_LOCAL, 'wb') as f_out:

    for linea in f_in:
        linea_limpia = linea.strip()
        if not linea_limpia:
            continue
        try:
            receta = json.loads(linea_limpia)
        except Exception:
            stats['json_errors'] += 1
            continue

        contador_total += 1
        stats['total_recetas_procesadas'] += 1

        try:
            req_id = receta.get('id')
            patient_id = dig(receta, 'subject', 'reference', default='').rsplit('/', 1)[-1]
            encounter_id = dig(receta, 'encounter', 'reference', default='').rsplit('/', 1)[-1]

            if patient_id not in pacientes_validos_ids:
                continue

            pharmacy_id = None
            for ident in receta.get('identifier', []):
                val = str(ident.get('value', ''))
                if val:
                    pharmacy_id = val
                    break

            requester_id = mapa_medicos.get(pharmacy_id, None)
            if not requester_id:
                stats['recetas_sin_medico_descartadas'] += 1
                continue

            delta = deltas_paciente.get(patient_id, pd.Timedelta(0))
            authored_on_val = receta.get('authoredOn')

            if authored_on_val and not pd.isnull(delta):
                dt_auth = pd.to_datetime(authored_on_val, errors='coerce')
                if not pd.isnull(dt_auth):
                    dt_reajustada = dt_auth - delta
                    if dt_reajustada.year != 2008:
                        continue
                    authored_on_val = dt_reajustada.isoformat()
            else:
                continue

            contador_exito += 1

            medication_id = dig(receta, 'medicationReference', 'reference', default='').rsplit('/', 1)[-1]
            medication_name = medicamentos.get(medication_id, 'unknown') if medication_id else 'unknown'

            if medication_name == 'unknown':
                concept_text = dig(receta, 'medicationCodeableConcept', 'text')
                if concept_text:
                    medication_name = concept_text
                else:
                    concept_display = dig(receta, 'medicationCodeableConcept', 'coding', 0, 'display')
                    concept_code = dig(receta, 'medicationCodeableConcept', 'coding', 0, 'code')
                    if concept_display: medication_name = concept_display
                    elif concept_code: medication_name = concept_code

            if medication_name == 'unknown':
                texto_dosis = dig(receta, 'dosageInstruction', 0, 'text', default='')
                route = dig(receta, 'dosageInstruction', 0, 'route', 'coding', 0, 'code', default='')
                if texto_dosis and (route == 'IV' or any(term in texto_dosis.lower() for term in ['bag', 'bottle', 'premix', 'soln', 'iv', 'infusion'])):
                    medication_name = f"IV Fluid / Mix: {texto_dosis}"
                else:
                    medication_name = f"Unknown Oral/Other: {texto_dosis}" if texto_dosis else "unknown"

            datos_encounter = encounters.get(encounter_id, {
                'encounter_class': 'unknown',
                'encounter_start': '',
                'encounter_end': '',
                'encounter_service_type': 'unknown'
            })

            dict_diags = diagnosticos.get(encounter_id, {})
            diags_texto = " | ".join([f"{cod}: {txt}" for cod, txt in dict_diags.items()]) if dict_diags else "No diagnosis recorded"

            dict_procs = procedimientos.get(encounter_id, {})
            procs_texto = " | ".join([f"{cod}: {txt}" for cod, txt in dict_procs.items()]) if dict_procs else "No procedure recorded"

            fue_administrado = req_id in administraciones

            dosage_text = dig(receta, 'dosageInstruction', 0, 'text', default='')
            route_code = dig(receta, 'dosageInstruction', 0, 'route', 'coding', 0, 'code', default='')
            timing_code = dig(receta, 'dosageInstruction', 0, 'timing', 'code', 'coding', 0, 'code', default='')
            dose_value = dig(receta, 'dosageInstruction', 0, 'dosageAndRate', 0, 'doseQuantity', 'value')
            dose_unit = dig(receta, 'dosageInstruction', 0, 'dosageAndRate', 0, 'doseQuantity', 'unit', default='')

            enc_start_val = datos_encounter['encounter_start']
            enc_end_val = datos_encounter['encounter_end']

            if enc_start_val and not pd.isnull(delta):
                dt_start = pd.to_datetime(enc_start_val, errors='coerce')
                if not pd.isnull(dt_start):
                    enc_start_val = (dt_start - delta).isoformat()
            if enc_end_val and not pd.isnull(delta):
                dt_end = pd.to_datetime(enc_end_val, errors='coerce')
                if not pd.isnull(dt_end):
                    enc_end_val = (dt_end - delta).isoformat()

            datos_paciente = pacientes.get(patient_id, {})

            fila_aplanada = {
                'IdMedicationRequest': req_id,
                'authoredOn': authored_on_val,
                'pharmacy_id': pharmacy_id,
                'order_provider_id': requester_id,
                'fue_administrado': fue_administrado,

                'encounter_id': encounter_id,
                'encounter_class': datos_encounter['encounter_class'],
                'encounter_start': enc_start_val,
                'encounter_end': enc_end_val,
                'encounter_service_type': datos_encounter['encounter_service_type'],
                'encounter_diagnoses': diags_texto,
                'encounter_procedures': procs_texto,

                'patient_id': patient_id,
                'subject_id': datos_paciente.get('subject_id'),
                'anchor_year_group': '2008 - 2010',
                'patient_gender': datos_paciente.get('patient_gender', 'unknown'),
                'patient_birthdate': datos_paciente.get('patient_birthdate', 'unknown'),

                'medication_id': medication_id,
                'medication_name': medication_name,
                'route_code': route_code,
                'timing_code': timing_code,
                'dose_value': dose_value,
                'dose_unit': dose_unit,
                'dosage_text': dosage_text
            }
            f_out.write(orjson.dumps(fila_aplanada) + b'\n')

        except Exception as e:
            continue

        if contador_total % 100000 == 0:
            print(f"    Procesadas {contador_total:,} recetas... (Válidas 2008: {contador_exito:,})")

print(f"\n🎉 ¡ETL local completado con éxito!")
print(f"   - Total de recetas válidas en 2008 guardadas: {contador_exito:,}")

print("\n📁 Copiando el archivo final a Google Drive...")
drive_output_path = os.path.join(BASE_DRIVE, 'medication_request_dataset_full.jsonl')
shutil.copy(OUTPUT_LOCAL, drive_output_path)
print(f"✅ ¡Archivo copiado y guardado correctamente en Google Drive: {drive_output_path}!")

 🚀 INICIANDO PROCESO ETL UNIFICADO (COHORTE 2008-2010 -> FILTRO 2008)

⏳ [1/5] Cargando el mapa de médicos...
✅ Mapa de médicos cargado: 17,779,025 registros indexados.

⏳ [2/5] Cargando el mapa de slots de pacientes...
✅ Mapa de slots cargado: 364,627 pacientes indexados.

⏳ [3/5] Cargando dimensiones en RAM (Pacientes, Encuentros, Diagnósticos, etc.)...
⏳ Cargando y seleccionando pacientes (Slot 2008-2010)...
✅ Pacientes cargados en RAM: 96,686
⏳ Cargando ingresos hospitalarios...
✅ Encuentros cargados: 323,465
⏳ Cargando diagnósticos...
⏳ Cargando procedimientos...
⏳ Cargando catálogo de medicamentos...
⏳ Cargando registros de administración...

⏳ [4/5] Analizando fechas mínimas por paciente (Optimizado con Caché)...
⏳ Analizando archivo grande por primera vez (usando orjson de alta velocidad)...
   -> Escaneadas 500,000 líneas...
   -> Escaneadas 1,000,000 líneas...
   -> Escaneadas 1,500,000 líneas...
   -> Escaneadas 2,000,000 líneas...
   -> Escaneadas 2,500,000 líneas...
   -> 

In [6]:
# ==========================================
# 6. INSPECCIÓN DE RESULTADOS
# ==========================================

import json
import os
import random

# Ruta de tu archivo resultante en Google Drive
archivo_resultado = '/content/drive/MyDrive/TFG/data/medication_request_dataset_full.jsonl'

if not os.path.exists(archivo_resultado):
    print(f"❌ No se encuentra el archivo en: {archivo_resultado}")
else:
    print(f"🔍 Inspeccionando registros aleatorios del dataset filtrado (Slot 2008-2010):\n")
    print("-" * 60)

    lineas_cargadas = []
    with open(archivo_resultado, 'rt', encoding='utf-8') as f:
        # Leemos las primeras 1,000 líneas (o menos si el archivo es más corto)
        for i in range(1000):
            linea = f.readline()
            if not linea:
                break
            lineas_cargadas.append((i + 1, linea.strip()))

    if not lineas_cargadas:
        print("⚠️ El archivo de resultados está vacío.")
    else:
        # Seleccionamos hasta 20 al azar
        muestra_aleatoria = random.sample(lineas_cargadas, min(20, len(lineas_cargadas)))

        # Ordenamos por su número de línea original para que se muestren en orden de aparición
        muestra_aleatoria.sort(key=lambda x: x[0])

        for idx, (num_linea, contenido_linea) in enumerate(muestra_aleatoria):
            try:
                datos = json.loads(contenido_linea)
                print(f"Muestra {idx + 1} (Línea original #{num_linea}):")
                print(json.dumps(datos, indent=4, ensure_ascii=False))
            except json.JSONDecodeError:
                print(f"Muestra {idx + 1} (Línea #{num_linea} - Error de decodificación JSON):")
                print(contenido_linea)
            print("-" * 60)

🔍 Inspeccionando registros aleatorios del dataset filtrado (Slot 2008-2010):

------------------------------------------------------------
Muestra 1 (Línea original #19):
{
    "IdMedicationRequest": "eb8dfd7d-391a-555a-800c-82e945392243",
    "authoredOn": "2008-01-14T16:46:57-05:00",
    "pharmacy_id": "53920939",
    "order_provider_id": "P11U9W",
    "fue_administrado": false,
    "encounter_id": "772fdad3-19ff-5da4-85f5-e369a910a506",
    "encounter_class": "EMER",
    "encounter_start": "2008-01-12T17:17:00-05:00",
    "encounter_end": "2008-01-19T18:18:00-05:00",
    "encounter_service_type": "OMED",
    "encounter_diagnoses": "V103: Personal history of malignant neoplasm of breast | 33829: Other chronic pain | 1985: Secondary malignant neoplasm of bone and bone marrow | 73313: Pathologic fracture of vertebrae | 72252: LUMB/LUMBOSAC DISC DEGEN | 71945: Pain in joint, pelvic region and thigh | 7384: Acquired spondylolisthesis | 53081: Esophageal reflux | 30000: Anxiety state, uns